
# 🧪 Inferencia, Tests de Hipótesis y A/B Testing — SOLUCIONES

Este notebook muestra **una posible solución** a los ejercicios del cuaderno de hipótesis y A/B testing.



## 0. Setup


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(123)

np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

print("Librerías listas.")



## 2. Test t de una muestra — Simulación inicial


In [ ]:

true_mean = 11
true_sd = 3
n = 50

muestra = rng.normal(true_mean, true_sd, size=n)
muestra = np.clip(muestra, 0, None)

print("Media muestral:", muestra.mean())
print("Desviación muestral:", muestra.std(ddof=1))



### Ejercicio 1 — Solución (cambiar μ₀)


In [ ]:

# Caso 1: mu0 = 10
mu0_1 = 10
xbar = muestra.mean()
s = muestra.std(ddof=1)
n = len(muestra)
se = s / np.sqrt(n)
t_stat_1 = (xbar - mu0_1) / se

print("Caso mu0=10 -> t =", t_stat_1)

# Caso 2: mu0 = 11 (más cerca de la media muestral)
mu0_2 = 11
t_stat_2 = (xbar - mu0_2) / se

print("Caso mu0=11 -> t =", t_stat_2)

# Comentario:
print("\nCuando mu0 está más cerca de la media muestral, el valor de t se acerca a 0.")



## 3. p-valor por simulación — Solución


In [ ]:

mu0 = 10  # mantenemos este caso para el ejemplo

xbar = muestra.mean()
s = muestra.std(ddof=1)
n = len(muestra)
se = s / np.sqrt(n)
t_stat = (xbar - mu0) / se

num_sims = 5000
t_stats_sim = []

for _ in range(num_sims):
    sim_data = rng.normal(mu0, s, size=n)
    xbar_sim = sim_data.mean()
    s_sim = sim_data.std(ddof=1)
    se_sim = s_sim / np.sqrt(n)
    t_sim = (xbar_sim - mu0) / se_sim
    t_stats_sim.append(t_sim)

t_stats_sim = np.array(t_stats_sim)
p_val_aprox = (1 + np.count_nonzero(np.abs(t_stats_sim) >= np.abs(t_stat))) / (num_sims + 1)

print("Estadístico t observado:", t_stat)
print("p-valor aproximado (bilateral):", p_val_aprox)

plt.figure()
plt.hist(t_stats_sim, bins=40)
plt.title("Distribución simulada de t bajo H0")
plt.xlabel("t* simulado"); plt.ylabel("frecuencia")
plt.show()



### Comentario de la solución (Ejercicio 2)

- Si el p-valor es pequeño (por ejemplo < 0.05), concluimos que la media es **significativamente distinta** de 10.  
- Al cambiar `true_mean`, se ve que:
  - Cuando `true_mean` está cerca de 10, el t observado y el p-valor tienden a ser "moderados".  
  - Cuando `true_mean` se aleja (por ejemplo 12), la media muestral suele estar más lejos de 10, t crece en valor absoluto y el p-valor se hace muy pequeño.



## 4. A/B Testing — Simulación de datos


In [ ]:

n_A = 80
n_B = 75

true_mean_A = 50
true_mean_B = 52
sd_common = 10

A = rng.normal(true_mean_A, sd_common, size=n_A)
B = rng.normal(true_mean_B, sd_common, size=n_B)

print("Media A:", A.mean(), "| Desv A:", A.std(ddof=1))
print("Media B:", B.mean(), "| Desv B:", B.std(ddof=1))



### 4.1. Estadístico de la diferencia de medias — Solución


In [ ]:

mean_A = A.mean()
mean_B = B.mean()
var_A = A.var(ddof=1)
var_B = B.var(ddof=1)

diff_mean = mean_B - mean_A
SE_diff = np.sqrt(var_A/n_A + var_B/n_B)
t_AB = diff_mean / SE_diff

print("Media A:", mean_A)
print("Media B:", mean_B)
print("Diferencia B - A:", diff_mean)
print("SE diff:", SE_diff)
print("t AB:", t_AB)



### Comentario (Ejercicio 3)

- Como en la simulación `true_mean_B` es mayor que `true_mean_A`, lo esperable es que `diff_mean` sea positiva.  
- Si acercamos `true_mean_B` a `true_mean_A`, la diferencia observada suele ser más pequeña y el valor de t se aproxima a 0.



## 5. p-valor aproximado por permutación — Solución


In [ ]:

combined = np.concatenate([A, B])
n_A = len(A)
n_B = len(B)

num_sims = 3000
diff_sims = []

for _ in range(num_sims):
    rng.shuffle(combined)
    A_sim = combined[:n_A]
    B_sim = combined[n_A:]
    diff_sim = B_sim.mean() - A_sim.mean()
    diff_sims.append(diff_sim)

diff_sims = np.array(diff_sims)
p_val_perm = (1 + np.count_nonzero(np.abs(diff_sims) >= np.abs(diff_mean))) / (num_sims + 1)

print("Diferencia observada B - A:", diff_mean)
print("p-valor aproximado (perm. bilateral):", p_val_perm)

plt.figure()
plt.hist(diff_sims, bins=40)
plt.title("Distribución de diferencias simuladas bajo H0 (A/B)")
plt.xlabel("diff_sim (B - A)"); plt.ylabel("frecuencia")
plt.show()



### Comentario (Ejercicio 4)

- Si el p-valor es < 0.05, solemos concluir que hay evidencia de que las medias de A y B son distintas.  
- Cuando `true_mean_B` es claramente mayor que `true_mean_A` y el tamaño de muestra es razonable, el p-valor tiende a ser muy pequeño.  
- Si bajamos el tamaño de muestra (n_A, n_B pequeños), la variabilidad aumenta y es más difícil detectar diferencias pequeñas → p-valor puede subir.



## 6. Resumen

Este cuaderno de soluciones muestra una forma razonable de:

- Calcular y comparar estadísticos t.  
- Aproximar p-valores por simulación para una muestra y para A/B tests.  
- Interpretar los resultados cualitativamente para la toma de decisiones en Data Science.
